In [ ]:
%%sql -r create_tables_result
USE DATABASE GATOR_DB;
USE SCHEMA PUBLIC;

CREATE OR REPLACE TABLE TESTING (
    SUCCESS_RATE NUMBER,
    START_DATE DATE,
    END_DATE DATE
);

CREATE OR REPLACE TABLE SANITATION (
    START_DATE DATE,
    END_DATE DATE,
    INFECTION_PREC NUMBER,
    SAN_METHOD VARCHAR,
    success_rate NUMBER,
    FOREIGN KEY (SUCCESS_RATE) REFERENCES TESTING(SUCCESS_RATE)
);

CREATE OR REPLACE TABLE Product (
    Product_id NUMBER,
    "DATE" NUMBER, -- to connect with the DIM DATE table as a surrogate key
    Product_Count NUMBER,
    Ingredients VARCHAR, -- foreign key
    STATUS VARCHAR,
    del_date DATE,
    success_rate NUMBER,
    SAN_START_DATE DATE,
    FOREIGN KEY (SUCCESS_RATE) REFERENCES TESTING(SUCCESS_RATE), --this is for connecting the success rates between tables
    FOREIGN KEY (SAN_START_DATE) REFERENCES SANITATION(START_DATE) --links the sanitation data for the table
);

CREATE OR REPLACE TABLE DIM_DATE (
    del_date DATE,
    week_day VARCHAR,
    month_12 NUMBER,
    quarter_4 NUMBER,
    hour_24 NUMBER, -- I included the 24 to symbolize that we are on a 24 hr clock -- not 12.
    year_num NUMBER
);

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE VIEW ML_FLAT_FILE AS
SELECT

    -- PRODUCT fields
    p.product_id,
    p.product_count,
    p.ingredients,
    p.status,

    -- all the data needed for the timings 
    p.del_date,
    d.week_day,
    d.month_12,
    d.quarter_4,
    d.hour_24,
    d.year_num,

    -- this will sync up the successrates even if they have unique names 
    p.success_rate AS product_success_rate,
    t.success_rate AS testing_success_rate,
    s.success_rate AS sanitation_success_rate,

    -- this is to make san_start_date in PRODUCT matches START_DATE in SANITATION
    p.san_start_date,
    s.start_date AS sanitation_start_date,
    s.end_date AS sanitation_end_date,
    s.infection_prec,
    s.san_method,

    --In my mind I wanted to have sanitation END_DATE match with testing START_DATE
    t.start_date AS testing_start_date,
    t.end_date AS testing_end_date

FROM PRODUCT p

--this is done to combine the dates together
JOIN DIM_DATE d     ON p.del_date      = d.del_date

-- Im am using this to combine the santitation date with the 
JOIN SANITATION s   ON p.san_start_date = s.start_date


JOIN TESTING t      ON p.success_rate   = t.success_rate
                   AND s.end_date        = t.start_date;   
